# L72 · Whisper 微调 — LoRA 低秩注入 vs 全参数，中文 / 方言数据适配实战

**目标**：用 HuggingFace `Trainer` + LoRA 在 LibriSpeech-clean-100h 子集上微调
`openai/whisper-small`，在 `test-clean` 上把 WER 压到 5% 以下。

🔗 **Aurora 连接**：本节的 mel 预处理路径与 `aurora.audio.features.mel_spectrogram`
对齐（均采用 25 ms 帧长、10 ms 步长、80 mel 通道）；实验结论将写入 blog 第2篇 `docs/blog/02_whisper_finetune.md`。

## 核心直觉

Whisper-small 有 244 M 参数，全量微调在单卡 A100 上也需要数小时且容易过拟合少量数据。LoRA 把每个注意力投影矩阵拆成"冻结的原始权重 + 两个低秩更新矩阵"，只训练后者（参数量 < 1%），梯度显存和训练时间同步缩减。`Seq2SeqTrainer` 把 gradient accumulation、mixed precision（fp16）和 checkpoint 打包成一行配置。

In [ ]:
# 需要 cloud GPU；依赖: transformers, peft, datasets, jiwer
# pip install transformers peft datasets jiwer
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model


## 1. LoRA（Low-Rank Adaptation）

对于权重矩阵 `W ∈ R^{d × k}`，LoRA 冻结 `W`，额外引入：

```
ΔW = A @ B       A ∈ R^{d × r},  B ∈ R^{r × k},  r << min(d, k)
```

前向传播变为 `y = (W + ΔW) x = W x + A @ B @ x`。初始化时 `A ~ N(0, σ²)`、`B = 0`，保证训练起点与原始模型等价。

对 Whisper-small（`d_model = 768`），自注意力中 `q_proj` 和 `v_proj` 各为 `768 × 768`。取 `r = 8`：原始可训练参数约 `2 × 768² ≈ 1.18 M`；LoRA 参数 `2 × 2 × 768 × 8 = 24 576`，**降幅 98%**。


In [ ]:
# 演示：加载模型并注入 LoRA，打印可训练参数比例
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,          # 缩放因子：ΔW 乘以 lora_alpha / r = 4
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 预期输出示例：trainable params: 24,576 || all params: 244,734,976 || trainable%: 0.010


## 2. HuggingFace `Seq2SeqTrainer`

`Seq2SeqTrainer` 在 `Trainer` 基础上增加 **生成式评估**（beam search），适配 encoder-decoder 模型。关键 `TrainingArguments` 参数：

| 参数 | 含义 |
|---|---|
| `per_device_train_batch_size` | 每卡 batch，影响显存；常取 16 |
| `gradient_accumulation_steps` | 累积 N 步再更新，等效扩大 batch |
| `fp16` | 混合精度；A100/V100 可开 |
| `predict_with_generate` | eval 时调用 `model.generate()` 算真实 WER |
| `generation_max_length` | 解码最长 token 数（LibriSpeech 句子取 225）|
| `save_steps` / `eval_steps` | 每 N 步存 checkpoint / 跑 eval |

实际有效 batch = `per_device_train_batch_size × gradient_accumulation_steps × num_gpus`。单卡 A100 40 GB 时，`batch=16, accumulation=2` 等效 batch=32。


In [ ]:
# 演示：加载数据集 + Processor，构造 DataCollator，配置 TrainingArguments
from datasets import load_dataset
from transformers import DataCollatorForSeq2Seq
import torch

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-small", language="English", task="transcribe"
)

# 只下载 train-clean-100（约 6 GB）和 test-clean 做快速演示
# 完整训练时去掉 split 限制
dataset = load_dataset(
    "librispeech_asr", "clean",
    split={"train": "train.100", "test": "test"},
    trust_remote_code=True,
)

def preprocess(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names, num_proc=4)

data_collator = DataCollatorForSeq2Seq(processor.tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-lora-librispeech",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,    # 有效 batch = 32
    learning_rate=1e-4,
    num_train_epochs=3,
    fp16=torch.cuda.is_available(),
    predict_with_generate=True,
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    evaluation_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,          # WER 越低越好
    report_to="tensorboard",
)
print("TrainingArguments 配置完成")


## 3. 评估指标：词错误率（WER）

```
WER = (S + D + I) / N
```

- `S`：替换（substitution）；`D`：删除（deletion）；`I`：插入（insertion）
- `N`：参考文本总词数

`jiwer` 库用动态规划在毫秒内算出编辑距离。LibriSpeech-test-clean 是朗读体英语，基线（zero-shot Whisper-small）约 **4–5% WER**；在 100h 子集上微调后目标 **< 5%**，甚至可逼近 2–3%。

注意：WER 对大小写和标点敏感，需在计算前做 **normalization**（`WhisperProcessor` 内置 `normalizer`）。


In [ ]:
# 演示：用 jiwer 计算 WER + 配置 compute_metrics 回调
import jiwer
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

normalizer = BasicTextNormalizer()

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    # 把 -100（padding）替换为 pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str  = processor.batch_decode(pred_ids,  skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    # 归一化
    pred_str  = [normalizer(s) for s in pred_str]
    label_str = [normalizer(s) for s in label_str]
    wer = jiwer.wer(label_str, pred_str)
    return {"wer": wer}

# 快速单元测试：WER 计算是否正确
ref  = ["the cat sat on the mat"]
hyp  = ["the cat sat on a mat"]          # 1 次替换，共 6 词
wer_val = jiwer.wer(ref, hyp)
assert abs(wer_val - 1/6) < 1e-6, f"WER 计算错误: {wer_val}"
print(f"✅ WER 单元测试通过：WER = {wer_val:.4f}（期望 {1/6:.4f}）")


## 参数实验：监控训练曲线，定位 overfitting

**可调参数与预期现象**：

| 参数 | 建议值 | 预期现象 |
|---|---|---|
| `r`（LoRA rank）| 4 / 8 / 16 | r 越大表达力越强，但过拟合风险更高；r=8 通常最优 |
| `lora_alpha` | `2r` 或 `4r` | alpha/r 是实际缩放倍数，增大等效提高学习率 |
| `num_train_epochs` | 3–5 | epoch 3 后 eval WER 通常不再下降，train loss 继续降→过拟合 |
| `learning_rate` | 1e-4 / 5e-5 | 1e-4 收敛快；5e-5 更稳但需更多 epoch |
| `lora_dropout` | 0.0 / 0.05 | dropout=0.05 对小数据集有正则效果 |

**监控方式**：TensorBoard 或 `trainer.state.log_history` 中的 `loss` 和 `eval_wer`。当 `eval_wer` 连续 2 个 eval_step 不下降而 `loss` 仍在降，即为 overfitting 起点。


In [ ]:
# 启动训练（需 GPU；在 CPU 上会极慢，仅作流程验证）
import torch

if torch.cuda.is_available():
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=processor.feature_extractor,
    )
    trainer.train()
    # 训练结束后查看最佳 checkpoint
    print("最佳 checkpoint：", trainer.state.best_model_checkpoint)
    print("最佳 WER：", trainer.state.best_metric)
else:
    print("⚠️  未检测到 GPU，跳过训练。请在 cloud GPU 环境（如 Colab A100）运行。")

# 可选：绘制 loss / WER 曲线
# import pandas as pd, matplotlib.pyplot as plt
# log_df = pd.DataFrame(trainer.state.log_history)
# fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# log_df.dropna(subset=["loss"]).plot("step", "loss", ax=axes[0], title="Train Loss")
# log_df.dropna(subset=["eval_wer"]).plot("step", "eval_wer", ax=axes[1], title="Eval WER")
# plt.tight_layout(); plt.show()


## 小结

`get_peft_model` 注入 LoRA 后，`Seq2SeqTrainer` 只更新 `q_proj`/`v_proj` 的低秩矩阵，在 3 个 epoch 内即可在 LibriSpeech-test-clean 上输出 WER < 5% 的转录结果。Aurora 模块 `aurora.audio.features.mel_spectrogram` 的 mel 预处理与`WhisperProcessor` 完全对齐，可复用于离线特征缓存以加速数据管道。下一节（M3-S5）将把微调好的 Whisper checkpoint 封装进 Aurora 推理接口，实现流式转录与时间戳对齐。
